# Layerwise stream RankMe over training — baseline vs layer-ablated

Ablated twin of experiments_anim_4: each animation shows the depth profile of stream
RankMe (centered) with the unablated sweep on the left and the layer-ablated run (writes
zeroed) on the right. Both panels are restricted to the runs' common checkpoints and share
one y-range.

The depth axis is **block outputs**: position ℓ is the stream after ℓ blocks have written,
so 0 is the embedding output, ℓ = L is `before_final_norm`, and `after_final_norm` closes
the axis. Blocks are numbered 1..L in the figure only — the code, the leaf names and the
`ablate_*` tags stay 0-indexed, so the panel title spells out both (`− blk4-7 (ℓ 5–8)`).

Dotted segments mark hops that are not a block doing work: the stretch an ablation skips
(every output inside it is a verbatim copy of the pre-ablation stream, so the copies are
hidden and one dotted line bridges output-to-output) and the closing hop into afn, which is
a normalization. The line stays solid for half a block past each measured point before it
breaks into dots, so every point keeps its normal line on both sides.

Pairs: pythia-1b (− blk3, the carrier), nanochat-d12 (− blk3-5), and the two OLMo-2 models
with their second quarter ablated (1B: − blk4-7, 7B: − blk8-15).

In [ ]:
import os, sys
import re
import shutil
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Which blocks are ablated is never inferred from the model — every call site passes its
# own tag, so one model can have as many ablated twins as there are ablate_* runs. Only
# the baseline sweep (a property of the model, not of the ablation) is looked up here.
BASE = lambda model: 'nanochat_samples' if model.startswith('nanochat') else 'block_representations_samples'

def _results(cfg, model):
    return np.load(f'data/results/{cfg}/results_{model}.npy', allow_pickle=True).item()

def measured_blocks(cfg, model):
    """The block indices that actually carry an attn.in stream in the results file."""
    hooks = {h for step in _results(cfg, model).values() for h in step}
    return sorted(int(m[1]) for h in hooks if (m := re.fullmatch(r'blk(\d+)\.attn\.in', str(h))))

COMMON = {}                                  # (cfg, model) -> the step set the pair shares

def register_pair(model, ablate):
    """(baseline cfg, ablated cfg) for the tag `ablate` (e.g. 'blk3', 'blk4-7'), registering
    the checkpoints both runs have so get_ys_aligned cuts each panel to them."""
    base, abl = BASE(model), f'ablate_{ablate}'
    shared = set(_results(base, model)) & set(_results(abl, model))
    COMMON[(base, model)] = COMMON[(abl, model)] = shared
    return base, abl

def get_ys_aligned(cfg, model, hook, yvar, *a, **kw):
    ys, steps = get_ys(cfg, model, hook, yvar, *a, **kw)
    allowed = COMMON.get((cfg, model))
    if ys is None or allowed is None:
        return ys, steps
    idx = [i for i, s in enumerate(steps) if s in allowed]
    return [ys[i] for i in idx], [steps[i] for i in idx]

In [ ]:
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_ys=get_ys_aligned, get_series_y=get_series_y,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)

In [ ]:
from IPython.display import display

# The depth axis reads as block OUTPUTS: position j is the stream after j blocks have written,
# so j = 0 is the embedding output, j = ℓ the output of block ℓ (1-indexed, figure-only — the
# code and the ablate_* tags stay 0-indexed), j = L is bfn, and afn closes the axis.
# blk{k}.attn.in is the stream entering code-block k = the output of the k-th block applied,
# which is why the sources need no shifting, only relabelling.

def parse_ablation(ablate):
    """'blk3' -> (3, 3), 'blk4-7' -> (4, 7): the ablated code-block indices (0-indexed)."""
    lo, _, hi = ablate.removeprefix('blk').partition('-')
    return int(lo), int(hi or lo)

def ablated_bridge(ablate, blocks):
    """The (first, last) depth positions the dotted bridge spans: the output of the last block
    before the ablation to the output of the first block after it ('blk3' -> (3, 5), 'blk4-7'
    -> (4, 9)). Every position in between is a verbatim copy of the pre-ablation stream —
    nothing was written into it — so the bridge hides them all. An ablation running to the
    last block has no post-ablation output and ends at bfn."""
    lo, hi = parse_ablation(ablate)
    x = lambda b: blocks.index(b) if b in blocks else len(blocks)   # past the last block -> bfn
    return x(lo), x(hi + 2)

def depth_ticks(L, stride=None):
    """Tick every `stride` block outputs (0 = embedding, L = bfn) plus afn at the end."""
    stride = stride or max(1, round(L / 8))
    ticks = list(range(0, L + 1, stride))
    return [(j, str(j)) for j in ticks + ([] if ticks[-1] == L else [L])] + [(L + 1, 'afn')]

def anim_layer_rankme_ablated(model, ablate, save_dir=None, fps=4):
    """`ablate` is the zeroed-write tag of the run to compare against, e.g. 'blk3' or
    'blk4-7' (data/results/ablate_<tag>); the depth axis comes from the results file. The
    ablated panel bridges the skipped stretch of depth with a dotted segment; both panels
    dot the closing hop into afn, which is a normalization rather than a block."""
    base, abl = register_pair(model, ablate)
    blocks = measured_blocks(base, model)
    L = len(blocks)
    def layers(src):
        ls = [(src, (f'blk{k}.attn.in', 'acts_centered'), f'blk{k}') for k in blocks]
        return ls + [(src, ('before_final_norm', 'acts_centered'), 'bfn'),
                     (src, ('after_final_norm', 'acts_centered'), 'afn')]
    lo, hi = parse_ablation(ablate)
    depth = {'kind': 'profile', 'ylog': True, 'xticks': depth_ticks(L),
             'xlabel': 'block output ℓ'}
    panels = [('rankme', layers(base), [model],
               depth | {'title': 'baseline', 'dotted_bridges': [(L, L + 1)]}),
              ('rankme', layers(abl), [model],
               depth | {'title': f'− {ablate}  (ℓ {lo + 1}' + (f'–{hi + 1})' if hi > lo else ')'),
                        'dotted_bridges': [ablated_bridge(ablate, blocks), (L, L + 1)]})]
    # materialize directly (mirrors animate_spectra) so the two panels can share one y-range
    spec = sa._materialize(panels, 2, fps, model, 'tokens', 'bottom',
                           f'Layerwise stream RankMe — {model}, baseline vs − {ablate}', None)
    lims = [p['ylim'] for p in spec['panels']]
    shared = (min(l[0] for l in lims), max(l[1] for l in lims))
    for p in spec['panels']:
        p['ylim'] = shared
    if save_dir:
        save = f'{save_dir}/layer_rankme_ablated_{model}_{ablate}.mp4'
        if shutil.which('ffmpeg') is None:
            sa._save_mp4_via_sbatch(spec, save)
        else:
            sa.render(spec, save)
    display(sa.render(spec, None))

In [ ]:
anim_layer_rankme_ablated('pythia-1b-deduped', 'blk3', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme_ablated('nanochat-d12', 'blk3-5', save_dir='analysis/figures/animations')

In [ ]:
# OLMo-2 1B, second quarter (blk4-7) zeroed.
anim_layer_rankme_ablated('OLMo-2-0425-1B', 'blk4-7', save_dir='analysis/figures/animations')

In [ ]:
# OLMo-2 7B, second quarter (blk8-15) zeroed.
anim_layer_rankme_ablated('OLMo-2-1124-7B', 'blk8-15', save_dir='analysis/figures/animations')